# MyAnimeList Score Prediction — Model 1: Linear Regression

This notebook implements a **Linear Regression** model to predict the user's anime rating (`my_score` on a 1-10 scale) based on user and anime interaction features produced by the AnimePref preprocessing pipeline.

Linear Regression is chosen as the first model because:
- It is **interpretable** — coefficients directly show which features increase or decrease predicted scores.
- It establishes a **baseline performance floor** for comparison with more complex models (Random Forest, Neural Network).
- It fits quickly, enabling efficient hyperparameter tuning via GridSearchCV with Ridge regularization.

**Task**: Predict `my_score` (numeric regression, 1–10) given user and anime features.

**Evaluation metrics**: 
- **MAE** (Mean Absolute Error) — average absolute prediction error in points
- **RMSE** (Root Mean Squared Error) — penalizes large errors more heavily
- **R^2** (Coefficient of Determination) — fraction of variance explained (0-1 scale)

In [6]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

INPUT_CANDIDATES = [
    Path("../processed-data/animepref_final_interactions.parquet"),
    Path("../processed-data/animepref_final_interactions.csv"),
    Path("../processed-data/animepref_final_interactions.parquet"),
    Path("processed-data/animepref_final_interactions.csv")
]

OUT_DIR = Path("../processed-data/splits")
OUT_DIR.mkdir(parents=True, exist_ok = True)

Load source dataset and validate schema.

In [9]:
src_path = None

for p in INPUT_CANDIDATES:
    if p.exists():
        src_path = p
        break

if src_path is None:
    raise FileNotFoundError("dataset is not found.")

if src_path.suffix.lower() == '.parquet':
    df = pd.read_parquet(src_path)
else:
    df = pd.read_csv(src_path, low_memory=False)

required_cols = ["username", "anime_id", "my_score", "last_updated_dt"]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded:", src_path.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Loaded: C:\Users\escan\Documents\02-academics\25-26\term_2\stintsy\01-actual-folder\STINTSYMajorOutput\processed-data\animepref_final_interactions.csv
Shape: (91209, 6)
Columns: ['username', 'anime_id', 'my_score', 'implicit_positive', 'my_status', 'last_updated_dt']


Filtering pipeline.

In [10]:
work = df.copy()

# Explicit ratings only
work["my_score"] = pd.to_numeric(work["my_score"], errors="coerce")
work = work[work["my_score"] >= 1].copy()

# Parse timestamp safely (no infer_datetime_format)
work["last_updated_dt"] = pd.to_datetime(work["last_updated_dt"], errors="coerce")

# Keep only rows with valid timestamp for chronological split
before_ts = len(work)
work = work.dropna(subset=["last_updated_dt"]).copy()

# Optional de-duplication: keep latest user-anime interaction
if {"username", "anime_id"}.issubset(work.columns):
    work = work.sort_values("last_updated_dt")
    work = work.drop_duplicates(subset=["username", "anime_id"], keep="last")

print("After rating filter + timestamp filter + dedup:")
print("Rows:", len(work))
print("Dropped invalid timestamp rows:", before_ts - len(work))

After rating filter + timestamp filter + dedup:
Rows: 91209
Dropped invalid timestamp rows: 0


Chronological split 70/15/15.

In [11]:
work = work.sort_values("last_updated_dt", kind="mergesort").reset_index(drop=True)

n = len(work)
i70 = int(n * 0.70)
i85 = int(n * 0.85)

train_df = work.iloc[:i70].copy()
val_df = work.iloc[i70:i85].copy()
test_df = work.iloc[i85:].copy()

print("Split sizes")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nDate ranges")
print("Train:", train_df["last_updated_dt"].min(), "->", train_df["last_updated_dt"].max())
print("Val  :", val_df["last_updated_dt"].min(), "->", val_df["last_updated_dt"].max())
print("Test :", test_df["last_updated_dt"].min(), "->", test_df["last_updated_dt"].max())

Split sizes
Train: (63846, 6)
Val: (13681, 6)
Test: (13682, 6)

Date ranges
Train: 1970-01-01 00:00:00 -> 2015-09-28 21:18:10
Val  : 2015-09-28 21:27:17 -> 2016-12-03 15:04:11
Test : 2016-12-03 15:21:35 -> 2018-05-20 23:47:25


Validation checks before export.

In [12]:
# No overlap by row index after slicing sorted frame
assert len(train_df) + len(val_df) + len(test_df) == len(work)

# Basic target distribution sanity
print("my_score describe")
print("\nTrain\n", train_df["my_score"].describe())
print("\nVal\n", val_df["my_score"].describe())
print("\nTest\n", test_df["my_score"].describe())

# Required columns present in all splits
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    miss = [c for c in required_cols if c not in part.columns]
    if miss:
        raise ValueError(f"{name} split missing columns: {miss}")

print("\nValidation checks passed.")

my_score describe

Train
 count    63846.000000
mean         7.200561
std          1.751241
min          1.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         10.000000
Name: my_score, dtype: float64

Val
 count    13681.000000
mean         6.941159
std          1.811949
min          1.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         10.000000
Name: my_score, dtype: float64

Test
 count    13682.000000
mean         6.928227
std          1.844823
min          1.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         10.000000
Name: my_score, dtype: float64

Validation checks passed.


Export exactly three CSV files.

In [13]:
train_csv = OUT_DIR / "animepref_train.csv"
val_csv = OUT_DIR / "animepref_val.csv"
test_csv = OUT_DIR / "animepref_test.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("Exported CSV files:")
print(train_csv.resolve())
print(val_csv.resolve())
print(test_csv.resolve())

Exported CSV files:
C:\Users\escan\Documents\02-academics\25-26\term_2\stintsy\01-actual-folder\processed-data\splits\animepref_train.csv
C:\Users\escan\Documents\02-academics\25-26\term_2\stintsy\01-actual-folder\processed-data\splits\animepref_val.csv
C:\Users\escan\Documents\02-academics\25-26\term_2\stintsy\01-actual-folder\processed-data\splits\animepref_test.csv


Readback check (quick integrity confirmation).

In [14]:
train_check = pd.read_csv(train_csv, low_memory=False)
val_check = pd.read_csv(val_csv, low_memory=False)
test_check = pd.read_csv(test_csv, low_memory=False)

print("Readback shapes")
print("Train:", train_check.shape)
print("Val:", val_check.shape)
print("Test:", test_check.shape)

Readback shapes
Train: (63846, 6)
Val: (13681, 6)
Test: (13682, 6)
